# 10 — Mars final package

Assemble the validated Mars-to-Earth outputs into a reproducible report. This notebook does not create fallback data and does not label a location as a confirmed analog.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUT = ROOT / 'mars_final_package'
OUT.mkdir(parents=True, exist_ok=True)

def find(name):
    hits = sorted(ROOT.rglob(name))
    return hits[0] if hits else None

candidate_path = find('refined_earth_matches.parquet') or find('earth_dust_environment_matches.parquet')
mars_validation_path = find('mars_candidate_validation_results.parquet')
distribution_path = find('mars_supported_environment_distributions.parquet')
if candidate_path is None:
    raise FileNotFoundError('No refined Earth candidate table found; run notebooks 08 and 09 first.')

candidates = pd.read_parquet(candidate_path)
if 'analog_score' in candidates.columns:
    candidates = candidates.sort_values('analog_score', ascending=False)
elif 'score' in candidates.columns:
    candidates = candidates.sort_values('score', ascending=False)
candidates.head(20).to_csv(OUT / 'top_earth_candidates.csv', index=False)

if {'latitude','longitude'}.issubset(candidates.columns):
    score_col = 'analog_score' if 'analog_score' in candidates else ('score' if 'score' in candidates else None)
    if score_col:
        fig, ax = plt.subplots(figsize=(12, 5))
        p = ax.scatter(candidates.longitude, candidates.latitude, c=candidates[score_col], s=12, cmap='viridis')
        ax.set(xlabel='Longitude', ylabel='Latitude', title='Ranked Earth compatibility candidates')
        fig.colorbar(p, ax=ax, label=score_col)
        fig.savefig(OUT / 'global_ranked_candidates.png', dpi=180, bbox_inches='tight')
        plt.close(fig)

summary = {
    'candidate_table': str(candidate_path),
    'mars_validation_table': str(mars_validation_path) if mars_validation_path else None,
    'mars_distribution_table': str(distribution_path) if distribution_path else None,
    'candidate_rows': int(len(candidates)),
    'top_locations_exported': int(min(20, len(candidates))),
    'interpretation': 'These regions are compatible with the tested Mars process signature under the measured variables; this is not proof of the same active process.',
    'limitations': ['Earth proxy variables are not Mars measurements', 'missingness and resolution can affect rank', 'independent field or higher-resolution validation is required'],
}
(OUT / 'run_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
(OUT / 'interpretation.md').write_text('# Mars-to-Earth final interpretation\n\n' + summary['interpretation'] + '\n\n## Limitations\n' + '\n'.join('- ' + x for x in summary['limitations']) + '\n', encoding='utf-8')
print(json.dumps(summary, indent=2))